# TC-WPN — Stage A: data preparation (CPU only)

**Run this with the accelerator set to `None`.** Nothing here needs a GPU, and
every minute of GPU time you spend on tokenisation is a minute you cannot spend
on training. Kaggle's GPU quota is roughly 30 h/week (it floats with demand);
CPU sessions do not draw on it.

At the end, commit this notebook. Its `/kaggle/working` output becomes a Kaggle
Dataset that Stage B attaches as input, so the expensive prep runs once.

### Before you run
Add your MIMIC datasets as inputs via **+ Add Input**. The notebook finds the
files itself, so the exact folder layout does not matter.

### What this produces
| file | what it is |
|---|---|
| `cohort_psych_mimic4.csv` | one row per note, labels from ICD only |
| `cohort_psych_mimic4_idx.csv` | after the index-time protocol |
| `audit_*.json` | the cohort audit for the paper's Data section |
| `pkl/*.pkl` | tokenised records |
| `plans/*.json` | frozen episode plans |
| `plans/leakage_certificate_*.json` | the 5,000-episode disjointness proof |

In [ ]:
# ---------------------------------------------------------------------------
# 1. Repo + dependencies.  Internet must be ON (Settings -> Internet).
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!git log --oneline -1
!pip install -q -r requirements.txt 2>&1 | tail -2

In [ ]:
# ---------------------------------------------------------------------------
# 2. Find the MIMIC files.  Adjust nothing unless this cell complains.
# ---------------------------------------------------------------------------
import sys, os
from pathlib import Path
sys.path.insert(0, "/kaggle/working/tcwpn_test/src")
from tcwpn.io_paths import resolve, has, MIMIC4, MIMIC3

INPUT = Path("/kaggle/input")
print("Available inputs:")
for d in sorted(INPUT.glob("*")):
    print("   ", d.name)

def find_base(keys, table):
    """Return the first input dir that contains every required table."""
    for d in sorted(INPUT.glob("*")):
        if all(has(d, *table[k]) for k in keys):
            return d
    return None

MIMIC4_PATH      = find_base(["patients", "admissions", "diagnoses"], MIMIC4)
MIMIC4_NOTE_PATH = find_base(["discharge"], MIMIC4)
MIMIC3_PATH      = find_base(["patients", "admissions", "diagnoses", "noteevents"], MIMIC3)

print("\nResolved:")
print("  MIMIC-IV hosp :", MIMIC4_PATH)
print("  MIMIC-IV note :", MIMIC4_NOTE_PATH)
print("  MIMIC-III     :", MIMIC3_PATH, "(optional, for the transfer arm)")

if MIMIC4_PATH is None or MIMIC4_NOTE_PATH is None:
    raise SystemExit(
        "Could not locate MIMIC-IV. Add the dataset as an input, or set "
        "MIMIC4_PATH / MIMIC4_NOTE_PATH by hand in this cell."
    )
os.environ["MIMIC_IV_DATASET_PATH"] = str(MIMIC4_PATH)
os.environ["MIMIC_IV_NOTE_DATASET_PATH"] = str(MIMIC4_NOTE_PATH)
if MIMIC3_PATH:
    os.environ["MIMIC_III_DATASET_PATH"] = str(MIMIC3_PATH)

## 3. Tests first

Your supervisor asked for `pytest` before anything touches data, and it costs
seconds. If this fails, stop — a red suite here means the leakage guarantees are
not holding and every number produced downstream is unverified.

In [ ]:
!PYTHONPATH=src python -m pytest tests/ -q

## 4. Build the cohort

`--arm psych` is the **primary** experiment: anxiety vs other psychiatric
illness. Labels come from ICD codes and the patient split is fixed before a
single note is loaded.

`--train-control-ratio 3` caps control patients in **train only** at 3× the
cases. Validation and test keep their natural prevalence, which is what the
clinical metrics need to mean anything.

In [ ]:
!python -m scripts.build_clean_cohort \
    --out /kaggle/working/data/clean \
    --source mimic4 --arm psych \
    --age-min 18 --age-max 50 \
    --max-notes-per-patient 8 \
    --train-control-ratio 3 \
    --split-salt tcwpn-clean-v1

## 5. Index-time protocol

This is the correction your supervisor asked for. Without it, a case patient's
notes can post-date the anxiety diagnosis, so the model may be reading
post-diagnosis documentation rather than detecting anything.

- `at_or_before` — **concurrent detection**. Notes up to the discharge of the
  index admission. The index admission's own summary is included, so explicit
  diagnosis language is available. This is the honest default; describe the task
  as concurrent, not predictive.
- `strictly_before` — **prospective detection**. Only prior admissions.

Run `at_or_before` first. Then read `differential_patient_retention_pp` in the
report: it tells you whether the policy removed cases and controls at different
rates. A large value means the surviving cohort is no longer the cohort you
defined.

In [ ]:
!python -m scripts.apply_index_time \
    --cohort   /kaggle/working/data/clean/cohort_psych_mimic4.csv \
    --patients /kaggle/working/data/clean/patients_psych_mimic4.csv \
    --policy at_or_before \
    --source mimic4 \
    --out /kaggle/working/data/clean/cohort_psych_mimic4idx.csv

### Optional: the prospective arm

Expect heavy attrition — a patient whose anxiety code appears at their first
admission has no prior record and disappears entirely. In MIMIC that is most
single-admission patients. Run it, read the retention numbers, and decide
whether the prospective arm is a viable second experiment or a limitation to
state in the paper. Either answer is publishable; guessing is not.

In [ ]:
!python -m scripts.apply_index_time \
    --cohort   /kaggle/working/data/clean/cohort_psych_mimic4.csv \
    --patients /kaggle/working/data/clean/patients_psych_mimic4.csv \
    --policy strictly_before \
    --source mimic4 \
    --out /kaggle/working/data/clean/cohort_psych_mimic4pro.csv

## 6. Audit

This table goes in the paper. Every split overlap must be `0` and every
text-derived filter must read `NO`.

In [ ]:
!python -m scripts.audit_cohort \
    --cohort /kaggle/working/data/clean/cohort_psych_mimic4idx.csv \
    --out    /kaggle/working/data/clean/audit_report_idx.json

## 7. Tokenise

Two variants from the **same** cohort file: unblinded, and `dx_meds` blinded.
Because they come from identical rows in identical order, they share one episode
plan and the robustness comparison is properly paired.

The stem is derived from the filename: `cohort_psych_mimic4idx.csv` gives stem
`psych_mimic4idx`. Keep that in mind for Stage B.

In [ ]:
COHORT = "/kaggle/working/data/clean/cohort_psych_mimic4idx.csv"
PKL    = "/kaggle/working/data/clean/pkl"

!python -m scripts.tokenize_cohort --cohort {COHORT} --out {PKL} --blind none
!python -m scripts.tokenize_cohort --cohort {COHORT} --out {PKL} --blind dx_meds

## 8. Freeze episode plans + leakage certificate

Plans are written to disk before any model exists, so ProtoNet, TC-WPN, TF-IDF
and the BERT probe all consume byte-identical episodes. That is what makes the
paired DeLong test in Stage B valid.

`--leakage-episodes 5000` is the stress test your supervisor asked for. It
must report a leakage rate of exactly 0 at every K.

In [ ]:
!python -m scripts.make_episode_plans \
    --pkl-dir /kaggle/working/data/clean/pkl \
    --stem psych_mimic4idx \
    --out /kaggle/working/data/clean/plans \
    --k 1 3 5 10 \
    --q-query 5 \
    --train-episodes 3000 \
    --eval-repeats 3 \
    --leakage-episodes 5000 \
    --seed 42

In [ ]:
# ---------------------------------------------------------------------------
# 9. Show the certificate and the audit -- these are paper artefacts.
# ---------------------------------------------------------------------------
import json, glob
for f in sorted(glob.glob("/kaggle/working/data/clean/plans/leakage_certificate_*.json")):
    print("="*70); print(f)
    print(json.dumps(json.load(open(f)), indent=2)[:2500])

for f in sorted(glob.glob("/kaggle/working/data/clean/*_index_report.json")):
    print("="*70); print(f)
    print(json.dumps(json.load(open(f)), indent=2)[:2000])

In [ ]:
# ---------------------------------------------------------------------------
# 10. Tidy the output so the committed dataset stays small.
#     The cohort CSVs carry full note text and can be large; keep them, they
#     are the reproducibility artefact, but drop the git checkout.
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test/.git
!du -sh /kaggle/working/data/clean/* | sort -h
print("\nNow: Save Version -> Save & Run All (Commit).")
print("Then in Stage B, add this notebook's output as an input dataset.")